# LoPy 影视级工作流 - Google Colab 交互运行笔记本 (MiniMax-H3 全链路版)

本笔记本支持在 **Google Colab** 环境下极速运行 MiniMax 官方最新发布的：
- **Stage 2 角色与首帧**：**MiniMax-H3-Image (33.1B DiT)** 图像生成与多模态角色/服装锁固控姿势
- **Stage 3 影视视频渲染**：**MiniMax H3 (Ref2VA)** 双流扩散视频生成与音画口型帧级对齐

> 💡 **核心运行建议**：
> 1. **显卡选型**：首选 **G4 (96GB Blackwell)** 或 **L4 (24GB Ada Lovelace)**，单图 4~30 秒，单视频 1.5~2.5 分钟。
> 2. **存储架构（动静分离）**：42GB 核心模型直接多线程拉取到本地 NVMe 临时盘（无 FUSE 延迟与限流）；Google Drive 仅用于**实时自动归档成片与输出资产**，防止因超时丢失成果！

In [ ]:
# [步骤 1] 检查 GPU 分配情况
!nvidia-smi

### [步骤 2] 挂载 Google Drive（自动实时归档成片，防意外丢失）
将输出目录 `runtime/outputs` 实时软链接到您的 Google 云端硬盘（`MyDrive/lopy_outputs`）。
所有生成的角色卡、首帧与视频成片，落盘即自动备份入云盘！

In [ ]:
from google.colab import drive
import os

USE_DRIVE = True # 强烈建议开启：自动归档成片
if USE_DRIVE:
    drive.mount('/content/drive')
    drive_out = '/content/drive/MyDrive/lopy_outputs'
    os.makedirs(drive_out, exist_ok=True)
    print(f"✅ Google Drive 挂载成功！输出将实时保存至: {drive_out}")
else:
    print("跳过 Google Drive 挂载，成片将仅保存在 Colab 临时磁盘。")

### [步骤 3] 克隆仓库 (dev-ai 分支) 并一键配置环境
拉取最新项目代码与规范工作区，并运行 `cloud/colab_setup.sh` 安装 ComfyUI master 与算子依赖。

In [ ]:
%cd /content
!rm -rf lopy-workflow-colab
!git clone -b dev-ai https://github.com/uljjmhn520/lopy-workflow-colab.git
%cd /content/lopy-workflow-colab

if USE_DRIVE:
    !rm -rf runtime/outputs
    !ln -sfn /content/drive/MyDrive/lopy_outputs runtime/outputs
    print("✅ runtime/outputs 已成功映射至 Google Drive！")

!bash cloud/colab_setup.sh

### [步骤 4] 极速多线程下载 MiniMax-H3-Image 图像模型权重
利用 Google Cloud 千兆带宽，通过 `aria2c` 直接下载到本地 NVMe 临时盘（耗时约 2~3 分钟）。
- 图像扩散模型：`minimax_h3_image_pruned_int8_convrot.safetensors` (~21GB)
- 文本/视觉编码器：`qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` (~15.7GB，图视共用)
- 图像 2D VAE：`minimax_h3_image_vae_fp16.safetensors` (~1.17GB)

In [ ]:
from pathlib import Path

MODELS_DIR = Path("/content/lopy-workflow-colab/models/H3/models")
(MODELS_DIR / "diffusion_models").mkdir(parents=True, exist_ok=True)
(MODELS_DIR / "text_encoders").mkdir(parents=True, exist_ok=True)
(MODELS_DIR / "vae").mkdir(parents=True, exist_ok=True)

image_files = [
    ("diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors"),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("vae/minimax_h3_image_vae_fp16.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors")
]

for rel_path, url in image_files:
    out_path = MODELS_DIR / rel_path
    if not out_path.exists():
        print(f"[下载中] {rel_path} ...")
        !aria2c -c -x 16 -s 16 --console-log-level=warn "{url}" -d "{out_path.parent}" -o "{out_path.name}"
    else:
        print(f"[已就绪] {rel_path}")

print("✅ MiniMax-H3-Image 图像模型准备就绪！")

### [步骤 5] 后台启动 ComfyUI 推理引擎

In [ ]:
import subprocess, time, urllib.request

cmd = (
    "python3 /content/ComfyUI/main.py "
    "--listen 127.0.0.1 --port 8188 "
    "--input-directory /content/lopy-workflow-colab/runtime/inputs "
    "--output-directory /content/lopy-workflow-colab/runtime/outputs "
    "--lowvram"
)

log_file = open("/content/comfy_server.log", "w")
proc = subprocess.Popen(cmd, shell=True, stdout=log_file, stderr=subprocess.STDOUT)
print("正在启动 ComfyUI 引擎...")

for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.status == 200:
            print("✅ ComfyUI 服务已成功启动！")
            break
    except Exception:
        time.sleep(2)
else:
    print("启动超时，请查看 /content/comfy_server.log 日志。")

### [步骤 6] 运行 MiniMax-H3-Image 纯文本生图（6步极速出片）
生成 9:16 竖屏 (768×1344) 角色立绘/首帧测试底图。首次加载模型约需 1.5 分钟，后续显存热载仅需 **4.2 秒**！

In [ ]:
# 使用 MiniMax-H3-Image 6步快速采样
!python3 tools/gen_image.py --model minimax-image --width 768 --height 1344 \
  --seed 1 \
  --prompt "Chinese high school girl Xiaoyun sitting in classroom, wearing blue and white school tracksuit uniform, black ponytail with bangs, sunset sunlight from left window, cinematic 500T, masterpiece" \
  --prefix h3_image_test

### [步骤 7] 首帧底图预览与效果检视

In [ ]:
from IPython.display import Image, display
import glob

imgs = sorted(glob.glob("/content/lopy-workflow-colab/runtime/outputs/*h3_image_test*.png"))
if imgs:
    latest_img = imgs[-1]
    print(f"最新生成图像: {latest_img}")
    display(Image(latest_img, width=400))
else:
    print("未检测到生成图像，请检查日志。")

### [步骤 8] 多图参考与角色/服装严格锁固控姿势
利用 MiniMax-H3 的 `MiniMaxH3ImageConditioning` 原生能力，以步骤 6 生成的晓云作为 `<Picture 1>`，
**精确锁死五官、发型与蓝白拼色立领校服，改变姿态为「站立举右手回答问题」**（实测耗时约 32 秒）。

In [ ]:
# 采用 MiniMax 官方推荐的「主体保留公式」，锁定服装与发型不变
!python3 tools/gen_image.py --model minimax-image \
  --ref runtime/outputs/h3_image_test_00001_.png \
  --seed 42 \
  --prefix xiaoyun_lock_cloth_42 \
  --prompt "Edit <Picture 1>. The clothing in <Picture 1> is fully preserved: Xiaoyun wears the exact identical Chinese school tracksuit jacket (loose navy blue and white sports jacket with high collar and front zipper). Her face and high black ponytail remain strictly identical to <Picture 1>. The only change is her pose: standing up beside her desk, raising her right hand high to answer a question with a polite smile."

In [ ]:
# 预览锁脸锁衣改姿势生成效果
lock_imgs = sorted(glob.glob("/content/lopy-workflow-colab/runtime/outputs/*xiaoyun_lock_cloth*.png"))
if lock_imgs:
    display(Image(lock_imgs[-1], width=400))
else:
    print("未检测到图像。")

### [步骤 9] 多线程下载 MiniMax H3 视频生成模型 (Ref2VA INT8 + 3D视频 VAE + 音频 VAE)
拉取视频专属核心模型（约 26.8GB，通过 `aria2c` 16 线程下载约需 2~3 分钟）。

In [ ]:
video_files = [
    ("diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
     "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7e75982b97cd5a41d2dcfa1904ee88d0686d6fd1/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"),
    ("vae/minimax_h3_video_vae_fp16.safetensors",
     "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7e75982b97cd5a41d2dcfa1904ee88d0686d6fd1/vae/minimax_h3_video_vae_fp16.safetensors"),
    ("vae/minimax_h3_audio_vae_fp32.safetensors",
     "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/7e75982b97cd5a41d2dcfa1904ee88d0686d6fd1/vae/minimax_h3_audio_vae_fp32.safetensors")
]

for rel_path, url in video_files:
    out_path = MODELS_DIR / rel_path
    if not out_path.exists():
        print(f"[下载中] {rel_path} ...")
        !aria2c -c -x 16 -s 16 --console-log-level=warn "{url}" -d "{out_path.parent}" -o "{out_path.name}"
    else:
        print(f"[已就绪] {rel_path}")

print("✅ MiniMax H3 视频生成全套模型就绪！")

### [步骤 10] 运行首个 9:16 影视级动态分镜（晓云首帧 + 台词口型对齐渲染）
以锁定的立领校服首帧作为画面起点，生成 3 秒动态短片（**768×1344 @ 24fps**，H.264 + AAC 声音口型同步）。

In [ ]:
import json

shot_spec = {
  "duration": 3.0,
  "size": [768, 1344],
  "seed": 42,
  "steps": 14,
  "style": "a realistic live-action film shot on 35mm, warm late-afternoon classroom window light, real skin texture and subtle believable acting",
  "setting": "a high school classroom with wooden desks and windows",
  "first_frame": "runtime/outputs/xiaoyun_lock_cloth_42_00001_.png",
  "subjects": [
    {
      "key": "yun",
      "desc": "Xiaoyun, a 17-year-old Chinese schoolgirl with neat black ponytail and straight bangs, wearing a blue and white school tracksuit uniform",
      "voice": "a gentle, clear 17-year-old Chinese girl's voice"
    }
  ],
  "audio": {
    "voice_ref": {
      "subject": "yun",
      "file": "tests/voice_test/out/test_line.wav"
    }
  },
  "shots": [
    {
      "camera": "The camera holds a static medium shot at eye level on a tripod.",
      "action": "Starting from <Picture 1>, Xiaoyun raises her right arm steadily from the wooden desk, lifting her hand with open palm toward shoulder height. Her head turns smoothly to her left, facing the teacher at the front of the classroom. Her dark eyes focus attentively, her eyebrows lifting with curious expression, and her lips part naturally into clear Chinese speech, smiling politely throughout.",
      "dialogue": [
        {
          "subject": "yun",
          "line": "老师，这道题我不太明白。",
          "lang": "Chinese",
          "after": "After speaking, she lowers her hand and listens attentively with a polite smile."
        }
      ]
    }
  ],
  "soundscape": "Quiet classroom ambient room tone.",
  "music": "N/A"
}

with open("runtime/shots/xiaoyun_shot.json", "w", encoding="utf-8") as f:
    json.dump(shot_spec, f, indent=2, ensure_ascii=False)

print("🚀 正在提交并渲染 H3 视频分镜（约需 2 分钟）...")
!python3 tools/h3_shot.py runtime/shots/xiaoyun_shot.json --out runtime/outputs/xiaoyun_video_test --force

### [步骤 11] 在线播放与检视渲染出的 9:16 动态成片

In [ ]:
import glob
from IPython.display import HTML, display

vids = sorted(glob.glob("/content/lopy-workflow-colab/runtime/outputs/h3_shot/*xiaoyun*.mp4"))
if vids:
    latest_vid = vids[-1]
    print(f"🎬 最新渲染成片已落盘: {latest_vid}")
    display(HTML(f'''
    <video width="384" height="672" controls autoplay loop style="border-radius: 8px; box-shadow: 0 4px 16px rgba(0,0,0,0.6);">
      <source src="{latest_vid}" type="video/mp4">
      您的浏览器不支持 HTML5 视频播放。
    </video>
    '''))
else:
    print("未检测到生成的视频，请检查日志。")